# EQRM vs ERM on Yelp Open Dataset

> **Runtime:** Colab Pro T4/A100  
> **Dataset:** Yelp Open Dataset (tar in Google Drive)  
> **Domain splits:** Geographic (city, 100+ domains) and Temporal (year-month, 100+ domains)  
> **Task:** Sentiment classification (1–5 stars) from review text  
> **Model:** DistilBERT fine-tuning  
> **Objective:** Test whether EQRM outperforms ERM when given abundant natural domains,
> contrasting with our Spawrious M2M results (2 domains, EQRM struggled).

## Cell 1 — GPU Check & Drive Mount

In [1]:
import torch, os, sys

assert torch.cuda.is_available(), "No GPU — switch runtime."
print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR    = '/content/drive/MyDrive/IFT6168/qrm/YELP'
RESULTS_DIR  = os.path.join(DRIVE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Drive dir: {DRIVE_DIR}")
print(f"Results:   {RESULTS_DIR}")

GPU: Tesla T4
Mounted at /content/drive
Drive dir: /content/drive/MyDrive/IFT6168/qrm/YELP
Results:   /content/drive/MyDrive/IFT6168/qrm/YELP/results


## Cell 2 — Install Dependencies

In [2]:
!pip install transformers accelerate --quiet

import transformers
print(f"Transformers: {transformers.__version__}")
print(f"PyTorch: {torch.__version__}")

Transformers: 5.0.0
PyTorch: 2.10.0+cu128


## Cell 3 — Extract & Load Yelp Data

The Yelp Open Dataset tar contains JSON files. We need:
- `yelp_academic_dataset_review.json` — review text, stars, date, business_id
- `yelp_academic_dataset_business.json` — city, state per business_id

We join them to get domain labels (city, year-month) for each review.

In [3]:
import tarfile, os, glob

WORK_DIR = '/content/yelp_data'
os.makedirs(WORK_DIR, exist_ok=True)

# Find the tar file in Drive
tar_candidates = glob.glob(os.path.join(DRIVE_DIR, '*.tar')) + \
                 glob.glob(os.path.join(DRIVE_DIR, '*.tar.gz')) + \
                 glob.glob(os.path.join(DRIVE_DIR, '*.tgz'))
print(f"Found tar files: {tar_candidates}")

# Check if already extracted
review_file = os.path.join(WORK_DIR, 'yelp_academic_dataset_review.json')
biz_file    = os.path.join(WORK_DIR, 'yelp_academic_dataset_business.json')

if os.path.exists(review_file) and os.path.exists(biz_file):
    print('Already extracted.')
else:
    assert len(tar_candidates) > 0, (
        f"No tar file found in {DRIVE_DIR}.\n"
        f"Contents: {os.listdir(DRIVE_DIR)}"
    )
    tar_path = tar_candidates[0]
    print(f"Extracting {tar_path}...")
    # Extract only the files we need
    with tarfile.open(tar_path, 'r:*') as tar:
        members = tar.getmembers()
        needed = [m for m in members if 'review' in m.name or 'business' in m.name]
        print(f"  Extracting {len(needed)} files...")
        for m in needed:
            # Flatten: extract to WORK_DIR regardless of internal path
            m.name = os.path.basename(m.name)
            tar.extract(m, WORK_DIR)

assert os.path.exists(review_file), f"Missing {review_file}"
assert os.path.exists(biz_file), f"Missing {biz_file}"

review_size = os.path.getsize(review_file) / 1e9
biz_size    = os.path.getsize(biz_file) / 1e6
print(f"\n✓ review.json: {review_size:.2f} GB")
print(f"✓ business.json: {biz_size:.1f} MB")

Found tar files: ['/content/drive/MyDrive/IFT6168/qrm/YELP/yelp_dataset.tar']
Extracting /content/drive/MyDrive/IFT6168/qrm/YELP/yelp_dataset.tar...
  Extracting 2 files...


/tmp/ipykernel_2912/2113567047.py:33: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(m, WORK_DIR)



✓ review.json: 5.34 GB
✓ business.json: 118.9 MB


## Cell 4 — Build Domain-Labeled DataFrame

Join reviews with business info to get city/state and parse dates.
This creates two domain columns: `domain_geo` (city) and `domain_time` (year-month).

In [4]:
import pandas as pd
import json
from collections import Counter

# ── Load business data (small, fits in memory) ──
print("Loading business data...")
businesses = {}
with open(biz_file, 'r') as f:
    for line in f:
        b = json.loads(line)
        businesses[b['business_id']] = {
            'city': b.get('city', 'unknown'),
            'state': b.get('state', 'unknown'),
        }
print(f"  {len(businesses)} businesses loaded")

# ── Load reviews (stream to avoid OOM on the 5GB file) ──
print("Loading reviews (streaming)...")
reviews = []
with open(review_file, 'r') as f:
    for i, line in enumerate(f):
        r = json.loads(line)
        biz = businesses.get(r['business_id'], {})
        reviews.append({
            'text':   r['text'][:1000],
            'stars':  r['stars'],
            'date':   r['date'],
            'city':   biz.get('city', 'unknown'),
            'state':  biz.get('state', 'unknown'),
        })
        if (i + 1) % 1_000_000 == 0:
            print(f"    {i+1:,} reviews loaded...")

df = pd.DataFrame(reviews)
del reviews

print(f"\nTotal reviews: {len(df):,}")

# ── Create domain columns ──
# Geographic: city, state, and city+state combo
df['domain_city']  = df['city'].str.lower().str.strip()
df['domain_state'] = df['state'].str.upper().str.strip()
df['domain_geo']   = df['domain_city'] + ', ' + df['domain_state']

# Temporal: year-month
df['domain_time'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)

# Label: 1-5 stars → 0-4
df['label'] = df['stars'].astype(int) - 1

print(f"Unique cities:       {df['domain_city'].nunique()}")
print(f"Unique states:       {df['domain_state'].nunique()}")
print(f"Unique city+state:   {df['domain_geo'].nunique()}")
print(f"Unique year-months:  {df['domain_time'].nunique()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

# ── Domain size diagnostics ──
import numpy as np
print("\n" + "="*50)
for col, label in [('domain_city', 'CITIES'), ('domain_state', 'STATES'),
                    ('domain_geo', 'CITY+STATE'), ('domain_time', 'MONTHS')]:
    counts = df[col].value_counts()
    print(f"\n{label} ({len(counts)} unique)")
    print(f"  Median:  {counts.median():.0f} reviews")
    print(f"  Mean:    {counts.mean():.0f} reviews")
    print(f"  ≥50:     {(counts >= 50).sum()} domains")
    print(f"  ≥200:    {(counts >= 200).sum()} domains")
    print(f"  ≥1000:   {(counts >= 1000).sum()} domains")
    print(f"  Top 5:   {list(counts.head(5).items())}")

Loading business data...
  150346 businesses loaded
Loading reviews (streaming)...
    1,000,000 reviews loaded...
    2,000,000 reviews loaded...
    3,000,000 reviews loaded...
    4,000,000 reviews loaded...
    5,000,000 reviews loaded...
    6,000,000 reviews loaded...

Total reviews: 6,990,280
Unique cities:       1238
Unique states:       27
Unique city+state:   1290
Unique year-months:  204

Label distribution:
label
0    1069561
1     544240
2     691934
3    1452918
4    3231627
Name: count, dtype: int64


CITIES (1238 unique)
  Median:  103 reviews
  Mean:    5646 reviews
  ≥50:     743 domains
  ≥200:    534 domains
  ≥1000:   320 domains
  Top 5:   [('philadelphia', 967906), ('new orleans', 635556), ('tampa', 455519), ('nashville', 451792), ('tucson', 405302)]

STATES (27 unique)
  Median:  51832 reviews
  Mean:    258899 reviews
  ≥50:     14 domains
  ≥200:    14 domains
  ≥1000:   14 domains
  Top 5:   [('PA', 1598960), ('FL', 1161545), ('LA', 761673), ('TN', 614388), (

## Cell 5 — Filter & Prepare Domain Splits

Keep only domains with ≥50 reviews so per-domain risk estimates are meaningful.
Split into train/val/test by holding out domains (not random samples).

In [6]:
import numpy as np

def prepare_split(df, domain_col, min_reviews=200, test_frac=0.2, val_frac=0.1, seed=42):
    """Split by holding out entire domains for test/val (random shuffle)."""
    counts = df[domain_col].value_counts()
    valid = counts[counts >= min_reviews].index.tolist()
    df_f = df[df[domain_col].isin(valid)].copy()
    rng = np.random.RandomState(seed)
    domains = sorted(valid)
    rng.shuffle(domains)
    n = len(domains)
    n_test = max(int(n * test_frac), 1)
    n_val  = max(int(n * val_frac), 1)
    train_d = set(domains[:n - n_test - n_val])
    val_d   = set(domains[n - n_test - n_val:n - n_test])
    test_d  = set(domains[n - n_test:])
    df_f['split'] = df_f[domain_col].apply(
        lambda d: 'train' if d in train_d else ('val' if d in val_d else 'test'))
    return df_f, train_d, val_d, test_d


def prepare_time_split(df, domain_col='domain_time', min_reviews=200,
                       test_frac=0.2, val_frac=0.1):
    """Split chronologically — train on past, test on future."""
    counts = df[domain_col].value_counts()
    valid = counts[counts >= min_reviews].index.tolist()
    df_f = df[df[domain_col].isin(valid)].copy()
    domains = sorted(valid)  # chronological order (YYYY-MM strings sort correctly)
    n = len(domains)
    n_test = max(int(n * test_frac), 1)
    n_val  = max(int(n * val_frac), 1)
    train_d = set(domains[:n - n_test - n_val])
    val_d   = set(domains[n - n_test - n_val:n - n_test])
    test_d  = set(domains[n - n_test:])
    df_f['split'] = df_f[domain_col].apply(
        lambda d: 'train' if d in train_d else ('val' if d in val_d else 'test'))
    return df_f, train_d, val_d, test_d


# ── State split (14 domains, few-domain regime) ──
df_state, state_tr, state_v, state_te = prepare_split(df, 'domain_state', min_reviews=2000)
print(f"STATE SPLIT")
print(f"  Train: {len(state_tr)}  Val: {len(state_v)}  Test: {len(state_te)} domains")
print(f"  Train: {(df_state['split']=='train').sum():,}  "
      f"Val: {(df_state['split']=='val').sum():,}  "
      f"Test: {(df_state['split']=='test').sum():,} reviews")

# ── City+State split (540 domains, many-domain regime) ──
df_geo, geo_tr, geo_v, geo_te = prepare_split(df, 'domain_geo', min_reviews=2000)
print(f"\nCITY+STATE SPLIT")
print(f"  Train: {len(geo_tr)}  Val: {len(geo_v)}  Test: {len(geo_te)} domains")
print(f"  Train: {(df_geo['split']=='train').sum():,}  "
      f"Val: {(df_geo['split']=='val').sum():,}  "
      f"Test: {(df_geo['split']=='test').sum():,} reviews")

# ── Temporal split (chronological, ~179 domains) ──
df_time, time_tr, time_v, time_te = prepare_time_split(df, min_reviews=2000)
print(f"\nTEMPORAL SPLIT (chronological)")
print(f"  Train: {len(time_tr)}  Val: {len(time_v)}  Test: {len(time_te)} domains")
print(f"  Train: {min(time_tr)}–{max(time_tr)}")
print(f"  Val:   {min(time_v)}–{max(time_v)}")
print(f"  Test:  {min(time_te)}–{max(time_te)}")
print(f"  Train: {(df_time['split']=='train').sum():,}  "
      f"Val: {(df_time['split']=='val').sum():,}  "
      f"Test: {(df_time['split']=='test').sum():,} reviews")

STATE SPLIT
  Train: 11  Val: 1  Test: 2 domains
  Train: 6,606,953  Val: 260,897  Test: 122,134 reviews

CITY+STATE SPLIT
  Train: 168  Val: 24  Test: 48 domains
  Train: 4,703,537  Val: 1,180,126  Test: 852,044 reviews

TEMPORAL SPLIT (chronological)
  Train: 120  Val: 16  Test: 33 domains
  Train: 2008-01–2017-12
  Val:   2018-01–2019-04
  Test:  2019-05–2022-01
  Train: 3,952,153  Val: 1,214,033  Test: 1,804,024 reviews


## Cell 6 — EQRM Loss (exact DomainBed implementation)

Faithful port of the EQRM class from `facebookresearch/DomainBed/domainbed/algorithms.py`,
contributed by @cianeastwood. Uses KDE with Gaussian kernels,
Gaussian-optimal bandwidth, and bisection to invert the CDF.
Gradients flow through `torch.distributions.Normal.cdf()` which is autograd-differentiable.

In [7]:
import torch
import torch.nn.functional as F
import numpy as np


class KDE_ICDF(torch.autograd.Function):
    """Invert the KDE CDF via bisection, with custom backward.

    Forward: find q such that F_KDE(q) = alpha, using bisection.
    Backward: use implicit function theorem:
        dq/d(risks) = -dF/d(risks) / (dF/dq)
    where dF/d(risks) and dF/dq come from the KDE CDF definition.
    """

    @staticmethod
    def forward(ctx, risks, alpha, bandwidth):
        """risks: (m,) tensor, alpha: float, bandwidth: scalar tensor."""
        m = len(risks)
        h = bandwidth.item()
        alpha_val = alpha.item() if isinstance(alpha, torch.Tensor) else alpha

        # Normal distribution for CDF/PDF
        normal = torch.distributions.Normal(0, 1)

        # KDE CDF: F(x) = (1/m) * sum_i Phi((x - r_i) / h)
        def kde_cdf_val(x):
            z = (x - risks) / bandwidth
            return normal.cdf(z).mean().item()

        # Bisection to find quantile
        lo = risks.min().item() - 4 * h
        hi = risks.max().item() + 4 * h

        for _ in range(32):
            mid = (lo + hi) / 2
            if kde_cdf_val(mid) < alpha_val:
                lo = mid
            else:
                hi = mid

        q = torch.tensor((lo + hi) / 2, dtype=risks.dtype, device=risks.device)

        # Save for backward
        ctx.save_for_backward(risks, bandwidth, q)
        ctx.alpha = alpha_val
        return q

    @staticmethod
    def backward(ctx, grad_output):
        risks, bandwidth, q = ctx.saved_tensors
        normal = torch.distributions.Normal(0, 1)
        m = len(risks)
        h = bandwidth

        z = (q - risks) / h

        # dF/dq = (1/m) * sum_i (1/h) * phi(z_i)   [phi = normal PDF]
        dF_dq = normal.log_prob(z).exp().mean() / h

        # dF/d(r_i) = -(1/m) * (1/h) * phi(z_i)
        dF_drisks = -(1.0 / m) * normal.log_prob(z).exp() / h

        # Implicit function theorem: dq/d(r_i) = -dF/d(r_i) / (dF/dq)
        dq_drisks = -dF_drisks / (dF_dq + 1e-10)

        return grad_output * dq_drisks, None, None


class EQRMLoss(torch.nn.Module):
    """EQRM from Eastwood et al. (NeurIPS 2022).

    Matches the DomainBed implementation:
    - KDE with Gaussian kernels
    - Gaussian-optimal bandwidth: h = (4/3m)^0.2 * sigma
    - Bisection to invert CDF (32 iterations)
    - Implicit function theorem for gradients
    """

    def __init__(self, alpha=0.75):
        super().__init__()
        self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float64))

    def forward(self, per_domain_risks):
        """per_domain_risks: tensor of shape (n_domains,)"""
        risks = per_domain_risks.double()
        m = len(risks)

        # Gaussian-optimal bandwidth (Eq. from Appendix C / Silverman)
        sigma = risks.std()
        h = (4.0 / (3.0 * m)) ** 0.2 * sigma

        if h < 1e-10:
            # All risks identical — just return mean
            return risks.mean().float()

        # Invert KDE CDF via bisection with custom backward
        quantile = KDE_ICDF.apply(risks, self.alpha, h)

        return quantile.float()


def compute_per_domain_risks(loss_per_sample, domain_ids):
    """Average the per-sample losses within each domain in the batch."""
    unique = domain_ids.unique()
    risks = []
    for d in unique:
        mask = (domain_ids == d)
        if mask.sum() >= 2:
            risks.append(loss_per_sample[mask].mean())
    if len(risks) < 2:
        # Not enough domains — fall back to ERM
        return loss_per_sample.mean()
    return torch.stack(risks)


# ── Quick test ──
print('Testing EQRM (exact DomainBed implementation)...')
test_risks = torch.tensor([0.1, 0.3, 0.5, 0.7, 0.9], requires_grad=True)
eqrm = EQRMLoss(alpha=0.75)
q = eqrm(test_risks)
q.backward()
print(f'  alpha=0.75  quantile={q.item():.4f}')
print(f'  gradients={[round(g, 4) for g in test_risks.grad.tolist()]}')

# Verify: higher alpha → higher quantile
test_risks2 = torch.tensor([0.1, 0.3, 0.5, 0.7, 0.9], requires_grad=True)
eqrm95 = EQRMLoss(alpha=0.95)
q95 = eqrm95(test_risks2)
q95.backward()
print(f'  alpha=0.95  quantile={q95.item():.4f}')
print(f'  gradients={[round(g, 4) for g in test_risks2.grad.tolist()]}')

assert q95.item() > q.item(), 'Higher alpha should give higher quantile!'
print('\n✓ EQRM loss verified: higher alpha → higher quantile → more weight on tail')

Testing EQRM (exact DomainBed implementation)...
  alpha=0.75  quantile=0.7707
  gradients=[0.0087, 0.0602, 0.2116, 0.3777, 0.3419]
  alpha=0.95  quantile=1.1073
  gradients=[0.0002, 0.004, 0.0443, 0.248, 0.7035]

✓ EQRM loss verified: higher alpha → higher quantile → more weight on tail


## Cell 7 — Dataset & DataLoader with Domain-Aware Batching

In [8]:
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import DistilBertTokenizer
from tqdm.auto import tqdm
import random

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


def pretokenize(texts, tokenizer, max_len=256, batch_size=1000):
    """Tokenize all texts upfront in batches. Returns (input_ids, attention_masks) tensors."""
    all_ids, all_masks = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="Pre-tokenizing"):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding='max_length', truncation=True,
                       max_length=max_len, return_tensors='pt')
        all_ids.append(enc['input_ids'])
        all_masks.append(enc['attention_mask'])
    return torch.cat(all_ids), torch.cat(all_masks)


class YelpDomainDataset(Dataset):
    def __init__(self, df, domain_col, max_len=256):
        texts = df['text'].tolist()
        self.labels = df['label'].values
        # Map domain strings to integer IDs
        domains = df[domain_col].tolist()
        unique = sorted(set(domains))
        self.domain_map = {d: i for i, d in enumerate(unique)}
        self.domain_ids = [self.domain_map[d] for d in domains]
        self.n_domains = len(unique)

        # Pre-tokenize everything once
        self.input_ids, self.attention_masks = pretokenize(
            texts, tokenizer, max_len=max_len
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
            'domain_id':      torch.tensor(self.domain_ids[idx], dtype=torch.long),
        }


class GroupBatchSampler(Sampler):
    """Sample batches that contain examples from multiple domains."""
    def __init__(self, domain_ids, batch_size, n_groups_per_batch=8):
        self.domain_ids = domain_ids
        self.batch_size = batch_size
        self.n_groups = n_groups_per_batch
        # Group indices by domain
        self.domain_to_indices = {}
        for i, d in enumerate(domain_ids):
            self.domain_to_indices.setdefault(d, []).append(i)
        self.domains = list(self.domain_to_indices.keys())
        self.per_group = max(batch_size // n_groups_per_batch, 1)

    def __iter__(self):
        # Shuffle within each domain
        pools = {d: list(idxs) for d, idxs in self.domain_to_indices.items()}
        for p in pools.values():
            random.shuffle(p)
        ptrs = {d: 0 for d in self.domains}

        for _ in range(len(self)):
            batch = []
            chosen = random.sample(self.domains, min(self.n_groups, len(self.domains)))
            for d in chosen:
                for _ in range(self.per_group):
                    if ptrs[d] >= len(pools[d]):
                        random.shuffle(pools[d])
                        ptrs[d] = 0
                    batch.append(pools[d][ptrs[d]])
                    ptrs[d] += 1
            yield batch[:self.batch_size]

    def __len__(self):
        return sum(len(v) for v in self.domain_to_indices.values()) // self.batch_size


print('✓ Dataset and sampler classes defined (with pre-tokenization)')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Dataset and sampler classes defined (with pre-tokenization)


## Cell 8 — Training & Evaluation Function

In [12]:
import time, json, os
from transformers import DistilBertForSequenceClassification
from tqdm.auto import tqdm


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    domain_correct, domain_total = {}, {}
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            y    = batch['label'].to(device)
            doms = batch['domain_id']
            preds = model(input_ids=ids, attention_mask=mask).logits.argmax(1)
            correct += (preds == y).sum().item()
            total += len(y)
            # Per-domain accuracy
            for d, p, t in zip(doms.tolist(), preds.cpu().tolist(), y.cpu().tolist()):
                domain_correct[d] = domain_correct.get(d, 0) + (p == t)
                domain_total[d]   = domain_total.get(d, 0) + 1
    avg_acc = correct / total
    domain_accs = [domain_correct[d] / domain_total[d]
                   for d in domain_total if domain_total[d] >= 5]
    domain_accs.sort()
    p10 = domain_accs[max(0, int(len(domain_accs) * 0.1))] if domain_accs else 0
    p25 = domain_accs[max(0, int(len(domain_accs) * 0.25))] if domain_accs else 0
    worst = domain_accs[0] if domain_accs else 0
    return {
        'avg_acc': avg_acc,
        'p10_acc': p10,
        'p25_acc': p25,
        'worst_acc': worst,
        'n_domains': len(domain_accs),
    }


def train_and_eval(split_df, domain_col, algo, alpha=0.75,
                   burnin_frac=0.1, n_epochs=1,
                   batch_size=64, lr=2e-5, max_len=128, device='cuda',
                   max_train=100_000, max_eval=15_000):
    name = f"{domain_col}__{algo}" + (f"_a{int(alpha*100)}" if algo == 'EQRM' else '')
    print(f"\n{'='*72}")
    print(f"  {name}")
    print(f"{'='*72}")

    # ── Balanced subsample: auto-compute per-domain cap from max_train ──
    train_df = split_df[split_df['split'] == 'train']
    val_df   = split_df[split_df['split'] == 'val']
    test_df  = split_df[split_df['split'] == 'test']

    for subset_name, sub_df, max_total in [('train', train_df, max_train),
                                            ('val', val_df, max_eval),
                                            ('test', test_df, max_eval)]:
        n_domains = sub_df[domain_col].nunique()
        max_per = max(max_total // max(n_domains, 1), 20)  # at least 20 per domain
        sub_df = sub_df.groupby(domain_col, group_keys=False).apply(
            lambda x: x.sample(min(len(x), max_per), random_state=42))
        if len(sub_df) > max_total:
            sub_df = sub_df.sample(max_total, random_state=42)
        if subset_name == 'train': train_df = sub_df
        elif subset_name == 'val': val_df = sub_df
        else: test_df = sub_df

    n_dom = train_df[domain_col].nunique()
    print(f'  Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
    print(f'  Train domains: {n_dom}  (~{len(train_df)//max(n_dom,1)} reviews/domain)')

    # ── Datasets ──
    train_ds = YelpDomainDataset(train_df, domain_col, max_len=max_len)
    val_ds   = YelpDomainDataset(val_df, domain_col, max_len=max_len)
    test_ds  = YelpDomainDataset(test_df, domain_col, max_len=max_len)

    print(f"  Train domains: {train_ds.n_domains}")

    # ── Loaders ──
    train_sampler = GroupBatchSampler(train_ds.domain_ids, batch_size, n_groups_per_batch=8)
    train_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=batch_size*2, shuffle=False, num_workers=2)

    # ── Model ──
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=5
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # ── EQRM setup ──
    eqrm_loss_fn = EQRMLoss(alpha=alpha) if algo == 'EQRM' else None
    total_steps = len(train_loader) * n_epochs
    burnin_steps = int(total_steps * burnin_frac) if algo == 'EQRM' else 0
    print(f"  Total steps: {total_steps}, Burn-in: {burnin_steps}")

    # ── Train ──
    model.train()
    global_step = 0
    t0 = time.time()

    for epoch in range(n_epochs):
        epoch_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}", leave=True):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            y    = batch['label'].to(device)
            doms = batch['domain_id'].to(device)

            logits = model(input_ids=ids, attention_mask=mask).logits
            per_sample_loss = F.cross_entropy(logits, y, reduction='none')

            if algo == 'EQRM' and global_step >= burnin_steps:
                domain_risks = compute_per_domain_risks(per_sample_loss, doms)
                if isinstance(domain_risks, torch.Tensor) and domain_risks.dim() > 0:
                    loss = eqrm_loss_fn(domain_risks)
                else:
                    loss = per_sample_loss.mean()
            else:
                loss = per_sample_loss.mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            global_step += 1

            if global_step % 500 == 0:
                print(f"    step {global_step}/{total_steps}  "
                      f"loss={loss.item():.4f}  "
                      f"({(time.time()-t0)/60:.1f} min)")

        print(f"  Epoch {epoch+1}/{n_epochs}  "
              f"avg_loss={epoch_loss/len(train_loader):.4f}")

    train_time = time.time() - t0

    # ── Evaluate ──
    print("  Evaluating...")
    val_metrics  = evaluate(model, val_loader, device)
    test_metrics = evaluate(model, test_loader, device)

    print(f"  Val:  avg={val_metrics['avg_acc']:.4f}  "
          f"p10={val_metrics['p10_acc']:.4f}  "
          f"worst={val_metrics['worst_acc']:.4f}")
    print(f"  Test: avg={test_metrics['avg_acc']:.4f}  "
          f"p10={test_metrics['p10_acc']:.4f}  "
          f"worst={test_metrics['worst_acc']:.4f}")

    # ── Save ──
    result = {
        'name': name, 'algo': algo, 'alpha': alpha,
        'domain_col': domain_col,
        'val': val_metrics, 'test': test_metrics,
        'train_time_min': round(train_time / 60, 1),
        'train_domains': train_ds.n_domains,
    }
    out_path = os.path.join(RESULTS_DIR, f"{name}.json")
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)

    # Save model
    model_path = os.path.join(RESULTS_DIR, f"{name}.pt")
    torch.save(model.state_dict(), model_path)
    print(f"  → Saved to {out_path}")

    return result


print('✓ Training function defined')

✓ Training function defined


## Cell 9 — Run All Experiments

2 domain splits × (1 ERM + 3 EQRM α values) = 8 runs total.

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

EXPERIMENTS = [
    # Geographic split
    {'split_df': df_geo,  'domain_col': 'domain_geo',  'algo': 'ERM',  'alpha': None},
    {'split_df': df_geo,  'domain_col': 'domain_geo',  'algo': 'EQRM', 'alpha': 0.75},
    {'split_df': df_geo,  'domain_col': 'domain_geo',  'algo': 'EQRM', 'alpha': 0.90},
    {'split_df': df_geo,  'domain_col': 'domain_geo',  'algo': 'EQRM', 'alpha': 0.95},
    # Temporal split
    {'split_df': df_time, 'domain_col': 'domain_time', 'algo': 'ERM',  'alpha': None},
    {'split_df': df_time, 'domain_col': 'domain_time', 'algo': 'EQRM', 'alpha': 0.75},
    {'split_df': df_time, 'domain_col': 'domain_time', 'algo': 'EQRM', 'alpha': 0.90},
    {'split_df': df_time, 'domain_col': 'domain_time', 'algo': 'EQRM', 'alpha': 0.95},
]

all_results = []
for exp in EXPERIMENTS:
    r = train_and_eval(
        split_df=exp['split_df'],
        domain_col=exp['domain_col'],
        algo=exp['algo'],
        alpha=exp.get('alpha', 0.75),
        n_epochs=1,
        batch_size=64,
        lr=2e-5,
        max_len=128,
        device=device,
    )
    all_results.append(r)

print(f"\n{'='*72}")
print(f"  ALL {len(all_results)} EXPERIMENTS COMPLETE")
print(f"{'='*72}")


  domain_geo__ERM


/tmp/ipykernel_7253/668284072.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df = train_df.groupby(domain_col, group_keys=False).apply(
/tmp/ipykernel_7253/668284072.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_df = val_df.groupby(domain_col, group_keys=False).apply(


  Train: 49,324  Val: 7,003  Test: 10,000


/tmp/ipykernel_7253/668284072.py:75: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_df = test_df.groupby(domain_col, group_keys=False).apply(


Pre-tokenizing:   0%|          | 0/50 [00:00<?, ?it/s]

...

  Total steps: 216, Burn-in: 21


Epoch 1/1:   0%|          | 0/216 [00:00<?, ?it/s]

  Epoch 1/1  avg_loss=1.4523
  Evaluating...


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd24c19c9a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive(): 
      ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
      Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd24c19c9a0>   
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^^^self._shutdown_workers()^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^if w.is_alive():^
^ ^ ^^ ^ ^   ^^^^^^^

  Val:  avg=0.5950  p10=0.4900  worst=0.4500
  Test: avg=0.6248  p10=0.5200  worst=0.4400
  → Saved to /content/drive/MyDrive/IFT6168/qrm/YELP/results/domain_time__EQRM_a95.json

  ALL 8 EXPERIMENTS COMPLETE


In [13]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Use a versioned results dir so previous results are preserved
RESULTS_DIR_V2 = os.path.join(DRIVE_DIR, 'results_v2')
os.makedirs(RESULTS_DIR_V2, exist_ok=True)

# Temporarily override RESULTS_DIR for this run
RESULTS_DIR = RESULTS_DIR_V2
print(f"Saving to: {RESULTS_DIR}")

EXPERIMENTS = [
    # State split (14 domains — few-domain regime)
    {'split_df': df_state, 'domain_col': 'domain_state', 'algo': 'ERM',  'alpha': None},
    {'split_df': df_state, 'domain_col': 'domain_state', 'algo': 'EQRM', 'alpha': 0.75},
    {'split_df': df_state, 'domain_col': 'domain_state', 'algo': 'EQRM', 'alpha': 0.95},
    # City+State split (168 domains — many-domain regime)
    {'split_df': df_geo,   'domain_col': 'domain_geo',   'algo': 'ERM',  'alpha': None},
    {'split_df': df_geo,   'domain_col': 'domain_geo',   'algo': 'EQRM', 'alpha': 0.75},
    {'split_df': df_geo,   'domain_col': 'domain_geo',   'algo': 'EQRM', 'alpha': 0.95},
    # Temporal split (120 domains — chronological)
    {'split_df': df_time,  'domain_col': 'domain_time',  'algo': 'ERM',  'alpha': None},
    {'split_df': df_time,  'domain_col': 'domain_time',  'algo': 'EQRM', 'alpha': 0.75},
    {'split_df': df_time,  'domain_col': 'domain_time',  'algo': 'EQRM', 'alpha': 0.95},
]

all_results = []
for exp in EXPERIMENTS:
    r = train_and_eval(
        split_df=exp['split_df'],
        domain_col=exp['domain_col'],
        algo=exp['algo'],
        alpha=exp.get('alpha', 0.75),
        n_epochs=1,
        batch_size=64,
        lr=2e-5,
        max_len=128,
        device=device,
    )
    all_results.append(r)

print(f"\n{'='*72}")
print(f"  ALL {len(all_results)} EXPERIMENTS COMPLETE")
print(f"{'='*72}")

Saving to: /content/drive/MyDrive/IFT6168/qrm/YELP/results_v2

  domain_state__ERM


/tmp/ipykernel_2912/3694503441.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_df = sub_df.groupby(domain_col, group_keys=False).apply(
/tmp/ipykernel_2912/3694503441.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_df = sub_df.groupby(domain_col, group_keys=False).apply(
/tmp/ipykernel_2912/3694503441.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This 

  Train: 99,990  Val: 15,000  Test: 15,000
  Train domains: 11  (~9090 reviews/domain)


Pre-tokenizing:   0%|          | 0/100 [00:00<?, ?it/s]

Pre-tokenizing:   0%|          | 0/15 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total steps: 1561, Burn-in: 156


Epoch 1/1:   0%|          | 0/1561 [00:00<?, ?it/s]

    step 500/1561  loss=1.7224  (6.4 min)
    step 1000/1561  loss=1.2300  (12.8 min)
    step 1500/1561  loss=1.3918  (19.2 min)
  Epoch 1/1  avg_loss=1.3141
  Evaluating...
  Val:  avg=0.7170  p10=0.6926  worst=0.6756
  Test: avg=0.7357  p10=0.7048  worst=0.6916
  → Saved to /content/drive/MyDrive/IFT6168/qrm/YELP/results_v2/domain_time__EQRM_a95.json

  ALL 9 EXPERIMENTS COMPLETE


## Cell 10 — Results Summary

In [10]:
import pandas as pd
import json, glob

rows = []
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, '*.json'))):
    with open(f) as fh:
        r = json.load(fh)
    rows.append({
        'name':          r['name'],
        'split':         r['domain_col'],
        'algo':          r['algo'],
        'alpha':         r.get('alpha', '-'),
        'train_domains': r['train_domains'],
        'test_avg':      round(r['test']['avg_acc'] * 100, 2),
        'test_p10':      round(r['test']['p10_acc'] * 100, 2),
        'test_p25':      round(r['test']['p25_acc'] * 100, 2),
        'test_worst':    round(r['test']['worst_acc'] * 100, 2),
        'time_min':      r['train_time_min'],
    })

df_results = pd.DataFrame(rows)
print('═' * 90)
print('  RESULTS  (test accuracies in %)')
print('═' * 90)
print(df_results.to_string(index=False))

══════════════════════════════════════════════════════════════════════════════════════════
  RESULTS  (test accuracies in %)
══════════════════════════════════════════════════════════════════════════════════════════
                 name       split algo  alpha  train_domains  test_avg  test_p10  test_p25  test_worst  time_min
 domain_geo__EQRM_a75  domain_geo EQRM   0.75            521     70.47     61.64     65.22       48.05       9.5
 domain_geo__EQRM_a90  domain_geo EQRM   0.90            521     67.95     59.21     63.38       37.04       9.5
 domain_geo__EQRM_a95  domain_geo EQRM   0.95            521     68.88     61.04     65.12       47.14       9.5
      domain_geo__ERM  domain_geo  ERM    NaN            521     70.42     62.86     65.75       48.15       9.3
domain_time__EQRM_a75 domain_time EQRM   0.75            140     61.80     49.00     57.00       45.00       2.7
domain_time__EQRM_a90 domain_time EQRM   0.90            140     62.32     50.00     57.00       46.00    

In [14]:
import pandas as pd
import json, glob

rows = []
for f in sorted(glob.glob(os.path.join(RESULTS_DIR_V2, '*.json'))):
    with open(f) as fh:
        r = json.load(fh)
    rows.append({
        'name':          r['name'],
        'split':         r['domain_col'],
        'algo':          r['algo'],
        'alpha':         r.get('alpha', '-'),
        'train_domains': r['train_domains'],
        'test_avg':      round(r['test']['avg_acc'] * 100, 2),
        'test_p10':      round(r['test']['p10_acc'] * 100, 2),
        'test_p25':      round(r['test']['p25_acc'] * 100, 2),
        'test_worst':    round(r['test']['worst_acc'] * 100, 2),
        'time_min':      r['train_time_min'],
    })

df_results = pd.DataFrame(rows)
print('═' * 90)
print('  RESULTS  (test accuracies in %)')
print('═' * 90)
print(df_results.to_string(index=False))

══════════════════════════════════════════════════════════════════════════════════════════
  RESULTS  (test accuracies in %)
══════════════════════════════════════════════════════════════════════════════════════════
                  name        split algo  alpha  train_domains  test_avg  test_p10  test_p25  test_worst  time_min
  domain_geo__EQRM_a75   domain_geo EQRM   0.75            168     71.79     66.99     69.87       63.46      19.9
  domain_geo__EQRM_a95   domain_geo EQRM   0.95            168     70.07     65.38     68.27       61.54      19.9
       domain_geo__ERM   domain_geo  ERM    NaN            168     71.25     66.03     68.59       63.46      19.8
domain_state__EQRM_a75 domain_state EQRM   0.75             11     69.84     69.63     69.63       69.63      19.9
domain_state__EQRM_a95 domain_state EQRM   0.95             11     68.31     68.23     68.23       68.23      20.0
     domain_state__ERM domain_state  ERM    NaN             11     69.68     69.55     69.55  

In [15]:
# Cell 11 — Auto-disconnect after completion
import time
print("All experiments complete. Results saved to Drive.")
print("Disconnecting runtime in 60 seconds...")
print("(Interrupt this cell to cancel)")

time.sleep(60)

from google.colab import runtime
runtime.unassign()

All experiments complete. Results saved to Drive.
Disconnecting runtime in 60 seconds...
(Interrupt this cell to cancel)


## Appendix — Why This Experiment

| Property | Spawrious M2M | Yelp Geographic | Yelp Temporal |
|---|---|---|---|
| Training domains | 2 | 100+ cities | 100+ year-months |
| Domain shift type | Engineered reversal | Natural geographic | Natural temporal |
| KDE quality | Degenerate (2 pts) | Well-estimated | Well-estimated |
| α interpretability | Imprecise | Meaningful | Meaningful |
| Evaluation | Single test env acc | Avg + P10 + worst | Avg + P10 + worst |

**Key metrics:** P10 (10th-percentile accuracy across domains) directly aligns with
EQRM's quantile objective. If EQRM improves P10 without sacrificing average accuracy,
it validates the probabilistic DG framework in a many-domain setting.